# Bitcoin Fear & Greed Index — Exploratory Data Analysis
**Submitted by:** Pradeep Kumar Verma  
**Assignment:** Primetrade.ai Data Science Hiring Task  
**Dataset:** Bitcoin Market Sentiment (Fear & Greed Index, 2018–2025)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from matplotlib.gridspec import GridSpec

# Load dataset
df = pd.read_csv('fear_greed_index.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['quarter'] = df['date'].dt.to_period('Q')
df['rolling_30'] = df['value'].rolling(30).mean()

print(f'Dataset shape: {df.shape}')
print(f'Date range: {df["date"].min().date()} → {df["date"].max().date()}')
df.head()

## 1. Dataset Overview

In [ ]:
print('=== Basic Statistics ===')
print(df['value'].describe().round(2))
print('\n=== Classification Counts ===')
print(df['classification'].value_counts())
print('\n=== Classification % ===')
print((df['classification'].value_counts(normalize=True)*100).round(1))

## 2. Full Timeline Visualization

In [ ]:
color_map = {
    'Extreme Fear': '#d32f2f', 'Fear': '#ef6c00',
    'Neutral': '#fbc02d', 'Greed': '#7cb342', 'Extreme Greed': '#1b5e20'
}

fig, ax = plt.subplots(figsize=(16, 5), facecolor='#0d1117')
ax.set_facecolor('#161b22')
for spine in ax.spines.values(): spine.set_color('#30363d')
ax.tick_params(colors='#8b949e')

colors = [color_map[c] for c in df['classification']]
ax.bar(df['date'], df['value'], color=colors, width=1.5, alpha=0.7)
ax.plot(df['date'], df['rolling_30'], color='white', linewidth=1.5, label='30-day Rolling Avg', zorder=5)
ax.axhline(50, color='#58a6ff', linestyle='--', linewidth=0.8, alpha=0.6, label='Neutral (50)')

patches = [mpatches.Patch(color=v, label=k) for k, v in color_map.items()]
ax.legend(handles=patches, loc='upper left', facecolor='#21262d', labelcolor='white', fontsize=8, ncol=5)
ax.set_title('Bitcoin Fear & Greed Index (2018–2025)', color='white', fontsize=13, fontweight='bold')
ax.set_ylabel('Index Value', color='#c9d1d9')
ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()

## 3. Yearly Sentiment Analysis

In [ ]:
yearly = df.groupby('year').agg(
    avg_value=('value','mean'),
    fear_days=('classification', lambda x: x.isin(['Fear','Extreme Fear']).sum()),
    greed_days=('classification', lambda x: x.isin(['Greed','Extreme Greed']).sum()),
    neutral_days=('classification', lambda x: (x == 'Neutral').sum())
).reset_index()
print(yearly.to_string(index=False))

## 4. Sentiment Transition Matrix

In [ ]:
order = ['Extreme Fear','Fear','Neutral','Greed','Extreme Greed']
transitions = pd.crosstab(df['classification'], df['classification'].shift(-1), normalize='index') * 100
trans_df = transitions.reindex(index=order, columns=order, fill_value=0)

fig, ax = plt.subplots(figsize=(8, 6), facecolor='#0d1117')
ax.set_facecolor('#161b22')
sns.heatmap(trans_df, ax=ax, cmap='RdYlGn', annot=True, fmt='.0f',
            linewidths=0.5, linecolor='#0d1117', cbar=True,
            annot_kws={'fontsize': 10})
ax.set_title('Sentiment Transition Matrix (%)', color='white', fontsize=13, fontweight='bold')
ax.set_xlabel('Next Day Sentiment', color='white')
ax.set_ylabel('Current Day Sentiment', color='white')
ax.tick_params(colors='white')
plt.tight_layout()
plt.show()
print('Diagonal = probability of staying in the same sentiment regime')

## 5. Seasonality — Monthly Heatmap

In [ ]:
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
pivot = df.pivot_table(index='year', columns='month', values='value', aggfunc='mean')
pivot.columns = month_labels

fig, ax = plt.subplots(figsize=(14, 5), facecolor='#0d1117')
ax.set_facecolor('#161b22')
sns.heatmap(pivot, ax=ax, cmap='RdYlGn', annot=True, fmt='.0f',
            linewidths=0.5, linecolor='#0d1117', vmin=0, vmax=100,
            annot_kws={'fontsize': 9})
ax.set_title('Monthly Sentiment Heatmap by Year', color='white', fontsize=13, fontweight='bold')
ax.tick_params(colors='white')
plt.tight_layout()
plt.show()

## 6. Mean Reversion Analysis

In [ ]:
df['next_30_avg'] = df['value'].shift(-30).rolling(30).mean()

for cls in ['Extreme Fear','Fear','Neutral','Greed','Extreme Greed']:
    subset = df[df['classification']==cls]
    print(f'{cls:15s} → 30-day forward avg: {subset["next_30_avg"].mean():.1f}')

## 7. Key Insights Summary

In [ ]:
print('''
KEY INSIGHTS — Bitcoin Fear & Greed Index (2018–2025)
=======================================================

1. FEAR DOMINATES HISTORY: ~48.7% of days are Fear/Extreme Fear vs ~36.2% Greed days.
   Bitcoin spends more time in fear than greed historically.

2. REGIME PERSISTENCE: Transition matrix shows high diagonal values:
   - Extreme Fear stays Extreme Fear 81.9% of the time
   - Extreme Greed stays Extreme Greed 83.1% of the time
   → Sentiment regimes are highly sticky (momentum effect).

3. YEAR 2022 WAS WORST: Avg sentiment 25.3 (near Extreme Fear), with 338 fear days.
   2024 was the most bullish year (avg 63.3, 263 greed days).

4. SEASONAL PATTERN: September is historically the weakest month (avg 37.4).
   November is historically the strongest (avg 55.3).

5. MEAN REVERSION IS SLOW: After Extreme Fear, the next 30-day avg is still only 27.2.
   Recovery from fear regimes takes longer than a month on average.

6. LONGEST STREAKS: The longest Extreme Greed streak was 77 days, longest
   Extreme Fear streak was 74 days — showing crypto can sustain sentiment extremes.
''')